# **Faster-RCNN Object Detection**

In [ ]:
!pip install pyyaml==5.1

import torch
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
# Install detectron2 that matches the above pytorch version
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
!pip install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="TdOKixM39Uh1tATTadka")
project = rf.workspace("mds-workspace-pgksy").project("people-detection-tbcmi")
dataset = project.version(1).download("coco")

In [ ]:
import detectron2
from detectron2.utils.logger import setup_logger
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

setup_logger()

In [ ]:
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog, MetadataCatalog

for d in ["my_dataset_train", "my_dataset_val"]:
    if d in DatasetCatalog.list(): DatasetCatalog.remove(d)
    if d in MetadataCatalog.list(): MetadataCatalog.remove(d)

register_coco_instances("my_dataset_train", {},
    "/content/people-detection-1/train/_annotations.coco.json",
    "/content/people-detection-1/train")

register_coco_instances(
    "my_dataset_val",
    {},
    "/content/people-detection-1/test/_annotations.coco.json",
    "/content/people-detection-1/test"
)

print("Datasets registered successfully.")

In [ ]:
dataset_dicts = DatasetCatalog.get("my_dataset_train")
for d in dataset_dicts:
    for ann in d["annotations"]:
        ann["category_id"] = int(ann["category_id"]) - 1

thing_classes = MetadataCatalog.get("my_dataset_train").thing_classes
MetadataCatalog.get("my_dataset_train").thing_classes = thing_classes

labels = {ann["category_id"] for d in dataset_dicts for ann in d["annotations"]}
labels

In [ ]:
import random

train_metadata = MetadataCatalog.get("my_dataset_train")
dataset_dicts = DatasetCatalog.get("my_dataset_train")

for d in random.sample(dataset_dicts, 3):
  img = cv2.imread(d["file_name"])
  visualizer = Visualizer(img[:, :, ::-1], metadata=train_metadata, scale=0.5)
  vis = visualizer.draw_dataset_dict(d)
  cv2_imshow(vis.get_image()[:, :, ::-1])

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [ ]:
from detectron2.engine import DefaultTrainer
import torch
torch.cuda.empty_cache()

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 300
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128

metadata = MetadataCatalog.get("my_dataset_train")
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(metadata.thing_classes)

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir output

In [ ]:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.DATASETS.TEST = ("my_dataset_test",)
predictor = DefaultPredictor(cfg)

In [ ]:
test_metadata = MetadataCatalog.get("my_dataset_test")

In [ ]:
from detectron2.utils.visualizer import ColorMode
import glob

for imageName in glob.glob('/content/people-detection-1/test/*jpg'):
  im = cv2.imread(imageName)
  outputs = predictor(im)
  v = Visualizer(im[:, :, ::-1],
                metadata=test_metadata,
                scale=0.8
                 )
  out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
  cv2_imshow(out.get_image()[:, :, ::-1])

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

evaluator = COCOEvaluator('my_dataset_val', output_dir='./output')
val_loader = build_detection_test_loader(cfg, 'my_dataset_val')

results = inference_on_dataset(trainer.model, val_loader, evaluator)
print(results)